In [38]:
from azure.ai.ml import MLClient, automl, Input
from azure.identity import DefaultAzureCredential
import mlflow
import os

In [4]:
ml_client =ml_client = MLClient.from_config(credential=DefaultAzureCredential())
mlflow_client = mlflow.MlflowClient()

Found the config file in: /config.json


In [45]:
# Registered Azure ML data asset
training_data =Input(
        type="mltable",
        path="azureml:transaction-train-data:1"
)
validation_data = Input(
        type="mltable",
        path="azureml:transaction-val-data:1"
)
test_data = Input(
        type="mltable",
        path="azureml:transaction-test-data:1"
)
TARGET_COLUMN = "is_fraud"

# Create the AutoML classification job
fraud_detection_job = automl.classification(
    training_data=training_data,
    validation_data=validation_data,
    test_data=test_data,
    target_column_name=TARGET_COLUMN,
    primary_metric="norm_macro_recall",
    experiment_name="Default",
    compute="durrain20031",
    enable_model_explainability=True,

)
fraud_detection_job.set_limits(
    max_trials=6,
    max_concurrent_trials=2,
)

# Submit the job
returned_job = ml_client.jobs.create_or_update(fraud_detection_job)
job_name = returned_job.name
print(f"Submitted job: {job_name}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Submitted job: gentle_dinner_4jdfgsnt12


In [2]:
job_name = 'gentle_dinner_4jdfgsnt12'

In [5]:
ml_client.jobs.get(job_name)

Experiment,Name,Type,Status,Details Page
Default,gentle_dinner_4jdfgsnt12,automl,Completed,Link to Azure Machine Learning studio


In [41]:
best_child_run_id = mlflow.get_run(job_name).data.tags['model_explain_best_run_child_id']
best_run = mlflow.get_run(best_child_run_id)

local_path = f"{os.path.dirname(os.getcwd())}/mlflow_artifacts"

# Download run's artifacts/outputs
local_path = mlflow_client.download_artifacts(
    best_run.info.run_id, "outputs", local_path
)
print(f"Artifacts downloaded in: {local_path}")
print(f"Artifacts: {os.listdir(local_path)}")

Artifacts downloaded in: /mnt/batch/tasks/shared/LS_root/mounts/clusters/durrain20031/code/FinRisk360/azureml/mlflow_artifacts/outputs
Artifacts: ['conda_env_v_1_0_0.yml', 'engineered_feature_names.json', 'env_dependencies.json', 'featurization_summary.json', 'generated_code', 'mlflow-model', 'model.pkl', 'pipeline_graph.json', 'run_id.txt', 'scoring_file_pbi_v_1_0_0.py', 'scoring_file_v_1_0_0.py', 'scoring_file_v_2_0_0.py']


In [53]:
mlflow.pyfunc.get_model_dependencies(local_path + "/mlflow-model/")

2026/09/28 05:37:40 INFO mlflow.pyfunc: To install the dependencies that were used to train the model, run the following command: '%pip install -r /mnt/batch/tasks/shared/LS_root/mounts/clusters/durrain20031/code/FinRisk360/azureml/mlflow_artifacts/outputs/mlflow-model/requirements.txt'.


'/mnt/batch/tasks/shared/LS_root/mounts/clusters/durrain20031/code/FinRisk360/azureml/mlflow_artifacts/outputs/mlflow-model/requirements.txt'

In [52]:
mlflow.pyfunc.load_model(local_path+'/mlflow-model/')

2026/09/28 05:36:22 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - anyio (current: 4.13.0, required: anyio==4.12.1)
 - arch (current: 8.0.0, required: arch==5.6.0)
 - attrs (current: 26.1.0, required: attrs==25.4.0)
 - azure-core (current: 1.41.0, required: azure-core==1.38.0)
 - azure-identity (current: 1.25.3, required: azure-identity==1.17.0)
 - azure-monitor-opentelemetry-exporter (current: 1.0.0b53, required: azure-monitor-opentelemetry-exporter==1.0.0b40)
 - azure-storage-blob (current: 12.27.1, required: azure-storage-blob==12.19.0)
 - azure-storage-queue (current: 12.17.0, required: azure-storage-queue==12.15.0)
 - azureml-automl-core (current: 1.62.0.post3, required: azureml-automl-core==1.61.0.post1)
 - azureml-automl-runtime (current: 1.62.0.post1, required: azureml-automl-runtime==1.61.0)
 - azureml-core (current: 1.61.0.post3, required: azureml-core==1.61.0.post1)
 - azureml-d

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [51]:
%pip install azureml-train-automl


  Installing build dependencies ... - \ | done
  Getting requirements to build wheel ... - done
  Preparing metadata (pyproject.toml) ... - done
INFO: pip is looking at multiple versions of azureml-train-automl-runtime to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of azureml-dataset-runtime[fuse,pandas] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of econml to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 122.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 116.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.9/987.

TypeError: get_portal_url() missing 1 required positional argument: 'run'